# From what exists to what we add: the fire-emissions MCP server

**The question we want answered:** *"How much CO₂, CO and methane did a wildfire release, and is its smoke visible in the air?"*

**What already exists** (organizers and the EVE MCP tool registry):

| Piece | What it gives |
|---|---|
| `effis` server: `geocode_place`, `get_effis_burnt_areas`, `compute_metrics` | **Detects the fire**: where, how big (ha), when, and burn severity |
| EVE model (fine-tuned from Mistral Small) and EVE's MCP gateway | The model that decides, and access to the registry servers |
| Other registry servers (gdacs, cdse, serpapi, ...) | Not used here |

**What we add**, our `fire-emissions` server, which **takes EFFIS output as its input**:

| Tool | Input (from EFFIS) | Output | Service behind it |
|---|---|---|---|
| `detect_burned_area` | `bbox`, `month` (used when EFFIS has no record) | burned ha, first/last/peak burn date | NASA MODIS Burned Area (MCD64A1), Planetary Computer API |
| `get_burn_fuel_types` | `bbox`, `firedate` | % forest, shrub, grass, crops (map from before the fire) | ESA WorldCover 2020/2021, Planetary Computer API |
| `estimate_fire_emissions` | `area_ha`, `bbox`, fuel shares | tonnes of CO₂, CO, CH₄, N₂O, PM2.5 | IPCC 2006 method (a calculation) |
| `get_smoke_signal` | `bbox`, `firedate` | CO / PM2.5 during the fire vs normal | CAMS via the Open-Meteo Air Quality API |

```
            ALREADY EXISTS                               WE ADD
"north Evia, Aug 2021" ─▶ geocode_place ─▶ get_effis_burnt_areas ─┬─▶ get_burn_fuel_types ─▶ estimate_fire_emissions
                                         (bbox, area_ha, firedate) └─▶ get_smoke_signal
                                                  │
                                       no fires (e.g. outside Europe)
                                                  ▼
                                    detect_burned_area (MODIS, global) ─▶ same three tools
```

**Plan of this notebook**

- **Part A:** look at what exists and call EFFIS step by step to detect the fire.
- **Part B:** feed EFFIS's output into our tools, at three levels (a Python function, the MCP server object, MCP over HTTP).
- **Part C:** before vs after. The model with EFFIS only, then with EFFIS plus our server.

Run the cells from top to bottom with the **.venv (Python 3.12)** kernel.

## 0. Setup

In [1]:
import asyncio
import importlib.util
import json
import logging
import os
import subprocess
import sys
import time
from pathlib import Path

import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Find the repo root (the folder that contains examples/fire-emissions-server).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "fire-emissions-server").exists())
SERVER_PY = ROOT / "examples" / "fire-emissions-server" / "server.py"

# EVE settings from .env: the model and the MCP gateway to the registry servers.
for line in (ROOT / ".env").read_text().splitlines():
    key, sep, value = line.partition("=")
    if sep and key.strip() and not key.startswith("#"):
        os.environ.setdefault(key.strip(), value.strip())
API = os.environ["EVE_API_BASE_URL"].rstrip("/")
AUTH = {"Authorization": f"Bearer {os.environ['EVE_API_KEY']}"}
MODEL = os.environ.get("EVE_LLM_MODEL", "jsc/alias-eve")

# Our server: imported as a Python module now (Part B), started over HTTP later (Part B, level 3).
spec = importlib.util.spec_from_file_location("server", SERVER_PY)
server = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server)
OUR_URL = "http://localhost:8000/mcp"

logging.disable(logging.INFO)   # hide request logs so the outputs stay readable

# Where each MCP server lives. "effis" is the registry's server, reached through EVE's gateway.
SERVERS = {
    "effis": (f"{API}/mcp/effis", AUTH),
    "fire-emissions": (OUR_URL, None),
}


async def mcp_list_tools(server_name):
    # Connect to one MCP server and return its tool list.
    url, headers = SERVERS[server_name]
    async with streamablehttp_client(url, headers=headers, timeout=180) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return (await session.list_tools()).tools


async def mcp_call(server_name, tool_name, arguments):
    # Connect to one MCP server, call one tool, return the result text.
    url, headers = SERVERS[server_name]
    async with streamablehttp_client(url, headers=headers, timeout=180) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
    return "".join(getattr(block, "text", "") for block in result.content)


def tool_json(text):
    # Decode a tool result. If the tool returned {"error": ...}, stop here and show that message
    # (otherwise the next line fails later with a confusing KeyError).
    data = json.loads(text)
    if isinstance(data, dict) and "error" in data:
        raise RuntimeError(f"Tool returned an error: {data['error']}  (often a busy external API: run the cell again)")
    return data


def pretty(text_or_dict, limit=1200):
    data = json.loads(text_or_dict) if isinstance(text_or_dict, str) else text_or_dict
    print(json.dumps(data, indent=1, ensure_ascii=False)[:limit])


print("EVE gateway:", API)
print("our server: ", SERVER_PY.relative_to(ROOT))

EVE gateway: https://dev.eve-chat.chat/api
our server:  examples/fire-emissions-server/server.py


# Part A: what already exists

## A1. The registry catalogue

EVE's backend hosts the registry servers. This is the list the agent can choose from **before** we add anything.

In [2]:
resp = httpx.get(f"{API}/mcp-servers", params={"limit": 50, "page": 1}, headers=AUTH, timeout=30)
resp.raise_for_status()
for s in resp.json().get("data", []):
    print(f"{s['name']:15s} {(s.get('description') or '').strip()[:95]}")

gdacs           GDACS is a cooperation framework between the United Nations and the European Commission. It inc
esa_moocs       Semantic search over ESA MOOC transcripts
trallie         Trallie: extract structured records from unstructured text or documents according to a requeste
serpapi         Live web search via SerpAPI: run a Google query and get back ranked results with titles, links 
eve_retrieval   Retrieval over the EVE knowledge base: fetch the passages most relevant to a query for groundin
eotdl           Earth Observation Training Data Lab (EOTDL): search the catalogue for training datasets and mod
eodashboard     EO Dashboard: browse and search Earth-observation narratives and themes, read narrative content
effis           European Forest Fire Information System (EFFIS): retrieve burnt-area records for a location or 
geocode         Geocoding tools (AgentCore runtime from eve-esa/mcp-tool-registry)
dummy_image     Returns a demo image (AgentCore runtime, artifact capture t

None of them estimates **emissions**, analyses **fuel types**, or checks **smoke**. The closest is `effis`, so let's see exactly what it offers.

## A2. The EFFIS server's tools

These schemas (name, parameters, description) are exactly what the model sees.

In [3]:
for tool in await mcp_list_tools("effis"):
    print("TOOL:", tool.name)
    print("  parameters:", list(tool.inputSchema["properties"]))
    print("  description:", tool.description.strip().splitlines()[0])
    print()

TOOL: geocode_place
  parameters: ['place_name', 'buffer_km', 'limit']
  description: Convert a place name to a bounding box for use with the fire tools.

TOOL: get_effis_burnt_areas
  parameters: ['bbox', 'max_features', 'date', 'buffer_km', 'save_dir', 'shapefile_dir']
  description: Detect burnt areas from EFFIS and return per-fire bboxes + metadata.

TOOL: compute_metrics
  parameters: ['bbox', 'fire_date', 'months_before', 'months_after', 'metrics', 'max_cloud_cover', 'resolution_m', 'width', 'height', 'min_mask_coverage', 'burn_threshold', 'min_aoi_coverage', 'recovery_months', 'save_dir']
  description: Compute spectral-index time series before and after a fire event,



## A3. Use EFFIS step by step to detect the fire

We do by hand what the agent will do later: place name → bounding box → burnt areas in August 2021.

In [4]:
# Step 1: place name -> bbox (registry tool)
geo = tool_json(await mcp_call("effis", "geocode_place", {"place_name": "Evia island, Greece"}))
island_bbox = geo["top_result"]["bbox"]
print("geocode_place -> Evia island bbox:", island_bbox)

# Step 2: burnt areas in that box in August 2021 (registry tool)
burnt = tool_json(await mcp_call("effis", "get_effis_burnt_areas",
                                  {"bbox": island_bbox, "date": "2021-08", "buffer_km": 0}))
print(f"\nget_effis_burnt_areas -> {burnt['total_features']} fires, largest first:")
for f in burnt["fires"][:4]:
    print(f"   {f['area_ha']:>9,.0f} ha   started {f['firedate'][:10]}   bbox {f['bbox']}")

geocode_place -> Evia island bbox: 22.8285,37.9489,24.5933,39.0396



get_effis_burnt_areas -> 8 fires, largest first:
      51,881 ha   started 2021-08-03   bbox 23.1766,38.7022,23.4614,39.0379
      10,175 ha   started 2021-08-16   bbox 23.2736,38.0612,23.4252,38.1865
       8,454 ha   started 2021-08-03   bbox 23.7489,38.0883,23.8984,38.2324
         175 ha   started 2021-08-22   bbox 24.3092,38.0605,24.3256,38.0862


The largest one is the north Evia fire. **These three values are everything our server needs as input.**

In [5]:
fire = burnt["fires"][0]
FIRE_BBOX = fire["bbox"]               # where:    west,south,east,north
FIRE_AREA_HA = fire["area_ha"]         # how big:  hectares
FIRE_DATE = fire["firedate"][:10]      # when:     YYYY-MM-DD

print("FIRE_BBOX    =", FIRE_BBOX)
print("FIRE_AREA_HA =", FIRE_AREA_HA)
print("FIRE_DATE    =", FIRE_DATE)
print("\nFields EFFIS returns for a fire:", list(fire))

FIRE_BBOX    = 23.1766,38.7022,23.4614,39.0379
FIRE_AREA_HA = 51881.0
FIRE_DATE    = 2021-08-03

Fields EFFIS returns for a fire: ['bbox', 'bbox_array', 'area_ha', 'commune', 'country', 'firedate']


## A4. The gap

EFFIS answers **where, how big and when** (and, with `compute_metrics`, how badly vegetation burned). Nothing in its output says **what kind of fuel burned, how many tonnes of gas went into the atmosphere, or whether people breathed the smoke**. That is what our server adds.

# Part B: our server on top of EFFIS

## Level 1: a tool is just a Python function

Open `server.py` and look at `get_burn_fuel_types`. It is an ordinary `async def` with:

- **type hints** (`bbox: str`), which become the parameter schema,
- a **docstring**, which becomes the description the model reads,
- a **JSON string** as the result.

`@mcp.tool()` registers the function on the server but leaves it unchanged, so we can call it directly, **with the bbox EFFIS gave us**.

In [6]:
result = await server.get_burn_fuel_types(bbox=FIRE_BBOX, fire_date=FIRE_DATE)   # EFFIS output -> our input

print(type(result))   # a str containing JSON
pretty(result)

<class 'str'>
{
 "bbox": "23.1766,38.7022,23.4614,39.0379",
 "worldcover_year": 2020,
 "map_choice": "2020 map: the land cover before a fire in 2021 or earlier.",
 "source": "ESA WorldCover 2020 v100 (10 m), Microsoft Planetary Computer",
 "box_hectares_by_class": {
  "tree_cover": 53205,
  "water": 27532,
  "grassland": 5231,
  "shrubland": 2127,
  "cropland": 1848,
  "bare_sparse": 1559,
  "built_up": 474,
  "herbaceous_wetland": 60
 },
 "fuel_shares_pct": {
  "tree_cover": 85.2,
  "grassland": 8.4,
  "shrubland": 3.4,
  "cropland": 3.0,
  "herbaceous_wetland": 0.1
 },
 "biome": "temperate",
 "note": "Shares describe the whole box, not only the burnt pixels inside it."
}


### Chaining: EFFIS → tool 1 → tool 2, and tool 3

```
EFFIS: FIRE_BBOX ─▶ get_burn_fuel_types ──fuel_shares_pct──▶ estimate_fire_emissions(FIRE_AREA_HA, FIRE_BBOX, ...)
EFFIS: FIRE_BBOX, FIRE_DATE ─▶ get_smoke_signal
```

In [7]:
# Step 1: what burned?
fuel = tool_json(await server.get_burn_fuel_types(bbox=FIRE_BBOX, fire_date=FIRE_DATE))
print("Step 1 -> fuel_shares_pct:", fuel["fuel_shares_pct"])

# Step 2: how much gas? Uses EFFIS's area plus the output of step 1.
emissions = tool_json(await server.estimate_fire_emissions(
    burned_area_ha=FIRE_AREA_HA,
    bbox=FIRE_BBOX,
    fuel_shares_pct=fuel["fuel_shares_pct"],
))
print("\nStep 2 -> tonnes per gas (low / central / high):")
for gas, r in emissions["emissions_tonnes"].items():
    print(f"   {gas:6s} {r['low']:>12,.0f} {r['central']:>12,.0f} {r['high']:>12,.0f}")
print("   CO2-eq central:", f"{emissions['co2_equivalent_tonnes']['central']:,.0f} t")
print("   car-years central:", f"{emissions['car_years_equivalent']['central']:,.0f}")

# Step 3: is the smoke visible in the air? Uses EFFIS's bbox and fire date.
smoke = tool_json(await server.get_smoke_signal(bbox=FIRE_BBOX, fire_date=FIRE_DATE))
print("\nStep 3 -> smoke peak vs normal:")
for name, p in smoke["pollutants"].items():
    print(f"   {name:16s} peak {p['fire_period_peak']:>7} ug/m3 = {p['peak_vs_baseline']}x baseline")

Step 1 -> fuel_shares_pct: {'tree_cover': 85.2, 'grassland': 8.4, 'shrubland': 3.4, 'cropland': 3.0, 'herbaceous_wetland': 0.1}

Step 2 -> tonnes per gas (low / central / high):
   CO2       1,090,000    2,180,000    3,640,000
   CO           73,000      146,000      244,000
   CH4           3,190        6,370       10,600
   N2O             178          356          594
   PM2_5         8,770       17,500       29,300
   CO2-eq central: 2,450,000 t
   car-years central: 473,000

Step 3 -> smoke peak vs normal:
   carbon_monoxide  peak  1909.0 ug/m3 = 12.3x baseline
   pm2_5            peak   103.3 ug/m3 = 7.5x baseline
   pm10             peak   143.2 ug/m3 = 6.1x baseline


### Errors are results too

A good tool never crashes the agent. A bad parameter returns a JSON `error` that the model can read and fix.

In [8]:
pretty(await server.get_burn_fuel_types(bbox="not-a-bbox"))
pretty(await server.estimate_fire_emissions(burned_area_ha=100, bbox=FIRE_BBOX, fuel_shares_pct={"forest": 100}))

{
 "error": "bbox must be 'west,south,east,north' in degrees, got 'not-a-bbox'."
}
{
 "error": "Unknown fuel types ['forest']. Use the names from get_burn_fuel_types.",
 "allowed": [
  "cropland",
  "grassland",
  "herbaceous_wetland",
  "mangroves",
  "moss_lichen",
  "shrubland",
  "tree_cover"
 ]
}


### When EFFIS has no record: `detect_burned_area`

EFFIS only knows fires it has already mapped, mostly in Europe. Our fourth tool detects burned area **anywhere**, from the NASA MODIS Burned Area product (500 m, monthly), counted on Planetary Computer's servers.

It is a fallback, so we only check it here on the Evia fire, where we can compare it with EFFIS:

In [9]:
modis = tool_json(await server.detect_burned_area(bbox=FIRE_BBOX, month=FIRE_DATE[:7]))
pretty(modis)
print(f"\nMODIS {modis['burned_area_ha']:,.0f} ha  vs  EFFIS {FIRE_AREA_HA:,.0f} ha")

{
 "bbox": "23.1766,38.7022,23.4614,39.0379",
 "month": "2021-08",
 "burned_area_ha": 50800.0,
 "burned_pct_of_box": 55.2,
 "source": "NASA MODIS Burned Area MCD64A1 v061 (500 m), Microsoft Planetary Computer",
 "note": "Approximate: 500 m pixels, all fires in the box together, fires under about 25 ha missed.",
 "first_burn_date": "2021-08-02",
 "last_burn_date": "2021-08-14",
 "peak_burn_date": "2021-08-06"
}

MODIS 50,800 ha  vs  EFFIS 51,881 ha


## Level 2: the same functions, through the MCP server object

`server.mcp` is the `FastMCP` server. For each tool it builds a **schema** from the type hints and docstring, the same kind of schema we saw for EFFIS in A2.

In [10]:
for tool in await server.mcp.list_tools():
    print("TOOL:", tool.name)
    print("  parameters:", json.dumps(tool.inputSchema["properties"]))
    print("  required:  ", tool.inputSchema.get("required", []))
    print("  description:", tool.description.strip().splitlines()[0])
    print()

TOOL: detect_burned_area
  parameters: {"bbox": {"title": "Bbox", "type": "string"}, "month": {"title": "Month", "type": "string"}}
  required:   ['bbox', 'month']
  description: Detect burned area anywhere in the world from satellite data, for one month.

TOOL: get_burn_fuel_types
  parameters: {"bbox": {"title": "Bbox", "type": "string"}, "fire_date": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Fire Date"}}
  required:   ['bbox']
  description: Break down the land cover (fuel types) inside a burnt-area bounding box.

TOOL: estimate_fire_emissions
  parameters: {"burned_area_ha": {"title": "Burned Area Ha", "type": "number"}, "bbox": {"title": "Bbox", "type": "string"}, "fuel_shares_pct": {"anyOf": [{"additionalProperties": {"type": "number"}, "type": "object"}, {"type": "null"}], "default": null, "title": "Fuel Shares Pct"}, "fire_date": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Fire Date"}}
  required:   ['burned_ar

Call a tool **by name, with a dict of parameters**. That `(name, arguments)` pair is the shape of every MCP call, for EFFIS and for ours.

In [11]:
content = await server.mcp.call_tool("get_smoke_signal", {"bbox": FIRE_BBOX, "fire_date": FIRE_DATE})
content = content[0] if isinstance(content, tuple) else content   # newer mcp versions return (content, structured)
print("MCP returns a list of content blocks:", [type(c).__name__ for c in content])
pretty(content[0].text, limit=600)

MCP returns a list of content blocks: ['TextContent']
{
 "location": {
  "lat": 38.87,
  "lon": 23.319
 },
 "fire_period": "2021-08-03 to 2021-08-13",
 "baseline_period": "2021-07-20 to 2021-08-02",
 "pollutants": {
  "carbon_monoxide": {
   "unit": "ug/m3",
   "baseline_mean": 154.8,
   "fire_period_mean": 323.2,
   "fire_period_peak": 1909.0,
   "peak_time_utc": "2021-08-06T21:00",
   "peak_vs_baseline": 12.3
  },
  "pm2_5": {
   "unit": "ug/m3",
   "baseline_mean": 13.7,
   "fire_period_mean": 27.4,
   "fire_period_peak": 103.3,
   "peak_time_utc": "2021-08-06T21:00",
   "peak_vs_baseline": 7.5
  },
  "pm10": {
   "unit": "ug/m3",
   "baseline


## Level 3: MCP over HTTP, the same way EFFIS is served

EFFIS runs as its own process behind `{API}/mcp/effis`. We start our server the same way, on `localhost:8000/mcp`.

In [12]:
proc = subprocess.Popen([sys.executable, str(SERVER_PY), "--transport", "http"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):   # wait until the server answers
    try:
        httpx.post(OUR_URL, json={}, timeout=1)
        break
    except httpx.HTTPError:
        time.sleep(0.5)
print("our server running, pid", proc.pid, "at", OUR_URL)

our server running, pid 351012 at http://localhost:8000/mcp


### 3a. The raw request: what actually travels over the wire

MCP is **JSON-RPC 2.0** over HTTP. A tool call is one POST with `method: "tools/call"`, the tool `name` and its `arguments`.

In [13]:
request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {"name": "estimate_fire_emissions",
               "arguments": {"burned_area_ha": FIRE_AREA_HA, "bbox": FIRE_BBOX}},
}
print("REQUEST:")
pretty(request)

resp = httpx.post(OUR_URL, json=request, timeout=120,
                  headers={"Accept": "application/json, text/event-stream"})

# The answer comes as a server-sent event: a line "data: {...json-rpc response...}"
data_line = next(line for line in resp.text.splitlines() if line.startswith("data:"))
response = json.loads(data_line[len("data:"):])
print("\nRESPONSE envelope keys:", list(response["result"].keys()))
print("Tool result inside it:")
pretty(response["result"]["content"][0]["text"], limit=700)

REQUEST:
{
 "jsonrpc": "2.0",
 "id": 1,
 "method": "tools/call",
 "params": {
  "name": "estimate_fire_emissions",
  "arguments": {
   "burned_area_ha": 51881.0,
   "bbox": "23.1766,38.7022,23.4614,39.0379"
  }
 }
}



RESPONSE envelope keys: ['content', 'structuredContent', 'isError']
Tool result inside it:
{
 "burned_area_ha": 51881,
 "biome": "temperate",
 "fuel_shares_pct_used": {
  "tree_cover": 81.8,
  "grassland": 11.9,
  "cropland": 4.4,
  "shrubland": 1.9
 },
 "emissions_tonnes": {
  "CO2": {
   "low": 1040000.0,
   "central": 2090000.0,
   "high": 3510000.0
  },
  "CO": {
   "low": 70200.0,
   "central": 140000.0,
   "high": 235000.0
  },
  "CH4": {
   "low": 3060.0,
   "central": 6120.0,
   "high": 10200.0
  },
  "N2O": {
   "low": 171.0,
   "central": 341.0,
   "high": 572.0
  },
  "PM2_5": {
   "low": 8420.0,
   "central": 16800.0,
   "high": 28200.0
  }
 },
 "co2_equivalent_tonnes": {
  "low": 1170000.0,
  "central": 2350000.0,
  "high": 3940000.0
 },
 "co2_equivalent_includes": [



### 3b. One client, two servers: EFFIS (registry) and ours (local)

`mcp_call` does not care where a server runs. The same function calls the hosted EFFIS server and our local one, which is how the agent will combine them.

In [14]:
# Registry server, through EVE's gateway
# Note: do not pass max_features=1 here. EFFIS cuts the list before sorting by size, so you would get
# a random small fire instead of the largest one.
burnt_again = tool_json(await mcp_call("effis", "get_effis_burnt_areas",
                                        {"bbox": island_bbox, "date": "2021-08", "buffer_km": 0}))
f = burnt_again["fires"][0]
print("effis          ->", f["area_ha"], "ha,", f["firedate"][:10], "bbox", f["bbox"])

# Our server, over HTTP, fed with EFFIS's output
fuel_http = tool_json(await mcp_call("fire-emissions", "get_burn_fuel_types",
                                     {"bbox": f["bbox"], "fire_date": f["firedate"]}))
print("fire-emissions ->", fuel_http["fuel_shares_pct"])

effis          -> 51881.0 ha, 2021-08-03 bbox 23.1766,38.7022,23.4614,39.0379


fire-emissions -> {'tree_cover': 85.2, 'grassland': 8.4, 'shrubland': 3.4, 'cropland': 3.0, 'herbaceous_wetland': 0.1}


# Part C: before vs after, letting the model decide

So far **we** chose the tools and the parameters. Now the EVE model does it. The loop is always the same:

```
1. send: question + tool schemas  ──▶  model
2. model answers with a tool call:   {"name": "...", "arguments": {...}}   (it does NOT run it)
3. WE run that tool through MCP and send the result back
4. repeat until the model answers with text instead of a tool call
```

We ask the same question twice: first with **EFFIS only**, then with **EFFIS + our server**.

In [15]:
BASE_PROMPT = (
    "You are an Earth Observation analyst who reports the climate impact of wildfires. "
    "Use the tools. Do not invent numbers. Cite the tool each fact came from. If the tools cannot answer part "
    "of the question, say so clearly. "
    "Never invent a bbox: if geocode_place returns a tiny box, geocode the wider area "
    "(for example the island or region) and pick the matching fire from the EFFIS results."
)
# Added only in C2, when our tools are available.
WORKFLOW = (
    " Workflow: get_effis_burnt_areas (buffer_km=0, date filter from the question; geocode_place first if you "
    "only have a place name) -> get_burn_fuel_types(bbox, firedate) -> estimate_fire_emissions(area_ha, bbox, "
    "fuel_shares_pct) -> get_smoke_signal(bbox, firedate). Give emissions as central value with low-high range. "
    "If get_effis_burnt_areas returns no fires (for example outside Europe), call detect_burned_area with a bbox "
    "around the place (geocode_place with buffer_km=50) and the month (YYYY-MM), then use its burned_area_ha, "
    "bbox and first_burn_date in the next steps."
)
QUESTION = ("How much CO2, carbon monoxide and methane did the August 2021 wildfire in north Evia, Greece "
            "release, and is the smoke visible in air-quality data? Short briefing with sources.")


async def build_tools(server_names, exclude=()):
    # Collect tool schemas from the given servers, in the format the chat API expects.
    route, llm_tools = {}, []
    for name in server_names:
        for tool in await mcp_list_tools(name):
            if tool.name in exclude:
                continue
            route[tool.name] = name
            llm_tools.append({"type": "function", "function": {
                "name": tool.name, "description": tool.description, "parameters": tool.inputSchema}})
    return route, llm_tools


async def step(messages, route, llm_tools):
    # One model turn: the model either asks for tools (we run them) or answers. Returns True when it answered.
    for attempt in range(6):   # EVE is shared by the whole room: wait and retry when it is busy (HTTP 429)
        resp = httpx.post(f"{API}/v1/chat/completions", headers=AUTH, timeout=300, json={
            "model": MODEL, "messages": messages, "tools": llm_tools, "temperature": 0.1, "max_tokens": 2048})
        if resp.status_code != 429:
            break
        wait = 15 * (attempt + 1)
        print(f"(EVE is busy, HTTP 429: waiting {wait} s)")
        await asyncio.sleep(wait)
    resp.raise_for_status()
    message = resp.json()["choices"][0]["message"]
    messages.append(message)

    calls = message.get("tool_calls") or []
    if not calls:
        print("MODEL FINAL ANSWER:\n")
        print(message.get("content"))
        return True
    if message.get("content"):
        print("model says:", message["content"])
    for call in calls:
        name = call["function"]["name"]
        args = call["function"]["arguments"]
        args = json.loads(args) if isinstance(args, str) else args
        print(f"MODEL CHOSE:  {name}   (server: {route.get(name, '?')})")
        print(f"  parameters: {json.dumps(args, ensure_ascii=False)}")
        if name in route:
            text = await mcp_call(route[name], name, args)          # WE run it, through MCP
        else:
            text = json.dumps({"error": f"Unknown tool {name}."})
        print(f"  result:     {text[:300]}{'...' if len(text) > 300 else ''}\n")
        messages.append({"role": "tool", "tool_call_id": call["id"], "content": text})
    return False


print("Question:", QUESTION)

Question: How much CO2, carbon monoxide and methane did the August 2021 wildfire in north Evia, Greece release, and is the smoke visible in air-quality data? Short briefing with sources.


## C1. Before: only what already exists (EFFIS)

In [16]:
route, llm_tools = await build_tools(["effis"])
print("tools the model can use:", list(route), "\n")

messages = [{"role": "system", "content": BASE_PROMPT}, {"role": "user", "content": QUESTION}]
for _ in range(8):
    if await step(messages, route, llm_tools):
        break

tools the model can use: ['geocode_place', 'get_effis_burnt_areas', 'compute_metrics'] 

(EVE is busy, HTTP 429: waiting 15 s)


(EVE is busy, HTTP 429: waiting 30 s)


MODEL CHOSE:  geocode_place   (server: effis)
  parameters: {"place_name": "north Evia, Greece", "buffer_km": 10}


  result:     {"place_name": "north Evia, Greece", "top_result": {"bbox": "23.3632,38.7088,23.5936,38.8890", "bbox_array": [23.3632, 38.7088, 23.5936, 38.889], "display_name": "North Evia \u039a\u03c4\u03b7\u03bc\u03b1\u03c4\u03bf\u03bc\u03b5\u03c3\u03b9\u03c4\u03b9\u03ba\u03b9, \u039c\u03b1\u03bd\u03c4\u03bf\u03...



model says: The geocode result for "north Evia, Greece" is a tiny point (an estate agent's office). I will geocode the wider area of Evia island and select the matching fire from EFFIS.
MODEL CHOSE:  geocode_place   (server: effis)
  parameters: {"place_name": "Evia island, Greece", "buffer_km": 0}


...

MODEL CHOSE:  get_effis_burnt_areas   (server: effis)
  parameters: {"bbox": "22.8285,37.9489,24.5933,39.0396", "date": "2021-08", "max_features": 5}


  result:     {"source": "EFFIS \u2014 Burnt Area Polygons", "date_filter": "2021-08", "data_source": "local_shapefile", "shapefile_dir": "/app/effis_layer", "total_features": 5, "saved_geojson": "/tmp/fire_maps/burnt_areas_2021-08_8df76a00.geojson", "buffer_km": 5.0, "fires": [{"bbox": "23.1188,38.6571,23.5191,3...



model says: The largest fire in Evia in August 2021 was the one that started on **August 3, 2021**, burning **51,881 hectares** (EFFIS — Burnt Area Polygons). This matches the major wildfire in north Evia.

I will now compute the emissions and check for smoke visibility in air quality data.
MODEL CHOSE:  compute_metrics   (server: effis)
  parameters: {"bbox": "23.1188,38.6571,23.5191,39.0829", "fire_date": "2021-08-03", "months_before": 1, "months_after": 1, "metrics": ["ndvi", "nbr", "bais2", "regrowth"], "recovery_months": [12, 24, 36, 48]}


  result:     {"ok": false, "error_type": "cdse_auth", "error": "CDSE credentials not set. Either send them as HTTP headers (X-CDSE-Client-Id / X-CDSE-Client-Secret) or set CDSE_CLIENT_ID / CDSE_CLIENT_SECRET env vars on the server."}



MODEL FINAL ANSWER:

I cannot compute emissions or air quality impacts because the tool requires authentication with CDSE credentials, which are not available. However, I can provide general information about the fire's emissions based on external sources and describe how to check for smoke visibility in air quality data.

---

### **Briefing: August 2021 North Evia Wildfire**

#### **1. Emissions**
The August 2021 wildfire in north Evia was one of Greece’s most destructive in recent years, burning **51,881 hectares** (EFFIS — Burnt Area Polygons). While I cannot compute exact emissions without CDSE authentication, estimates from similar Mediterranean fires suggest:

- **CO₂ emissions**: Wildfires in Greece typically release **10–30 tonnes of CO₂ per hectare burned**, depending on vegetation type. For 51,881 ha, this could range from **500,000 to 1.5 million tonnes of CO₂**.
- **Carbon monoxide (CO)**: Mediterranean fires emit **~50 kg of CO per hectare**. For this fire, that could be 

The model can find the fire but has **no tool for gases or smoke**. It either says it cannot answer that part, or it falls back on general knowledge, which is exactly the gap.

## C2. After: EFFIS + our fire-emissions server

Run the second cell **again and again** to follow the model one decision at a time. Each step shows which **server** the tool came from.

In [17]:
# compute_metrics needs CDSE credentials (see C1), so we leave it out.
route, llm_tools = await build_tools(["effis", "fire-emissions"], exclude=["compute_metrics"])
print("tools the model can use:")
for tool, server_name in route.items():
    print(f"   {server_name:15s} -> {tool}")

messages = [{"role": "system", "content": BASE_PROMPT + WORKFLOW}, {"role": "user", "content": QUESTION}]
done = False

tools the model can use:
   effis           -> geocode_place
   effis           -> get_effis_burnt_areas
   fire-emissions  -> detect_burned_area
   fire-emissions  -> get_burn_fuel_types
   fire-emissions  -> estimate_fire_emissions
   fire-emissions  -> get_smoke_signal


In [18]:
# Run this cell repeatedly to follow the model one decision at a time.
if not done:
    done = await step(messages, route, llm_tools)
else:
    print("Finished. Run the next cell to do the remaining steps automatically.")

MODEL CHOSE:  geocode_place   (server: effis)
  parameters: {"place_name": "north Evia, Greece", "buffer_km": 0}


  result:     {"place_name": "north Evia, Greece", "top_result": {"bbox": "23.4783,38.7989,23.4784,38.7990", "bbox_array": [23.4783, 38.7989, 23.4784, 38.799], "display_name": "North Evia \u039a\u03c4\u03b7\u03bc\u03b1\u03c4\u03bf\u03bc\u03b5\u03c3\u03b9\u03c4\u03b9\u03ba\u03b9, \u039c\u03b1\u03bd\u03c4\u03bf\u03...



In [19]:
# Or let it finish by itself.
for _ in range(10):
    if done:
        break
    done = await step(messages, route, llm_tools)

model says: The geocode result for "north Evia, Greece" is a tiny point (23.4783,38.7989,23.4784,38.7990), which is not useful for fire detection. I will geocode the wider area "Evia island" instead.
MODEL CHOSE:  geocode_place   (server: effis)
  parameters: {"place_name": "Evia island, Greece", "buffer_km": 0}


  result:     {"place_name": "Evia island, Greece", "top_result": {"bbox": "22.8285,37.9489,24.5933,39.0396", "bbox_array": [22.8285, 37.9489, 24.5933, 39.0396], "display_name": "\u0395\u03cd\u03b2\u03bf\u03b9\u03b1, \u0391\u03c0\u03bf\u03ba\u03b5\u03bd\u03c4\u03c1\u03c9\u03bc\u03ad\u03bd\u03b7 \u0394\u03b9\u03bf...



MODEL CHOSE:  get_effis_burnt_areas   (server: effis)
  parameters: {"bbox": "22.8285,37.9489,24.5933,39.0396", "date": "2021-08", "buffer_km": 0}


  result:     {"source": "EFFIS \u2014 Burnt Area Polygons", "date_filter": "2021-08", "data_source": "local_shapefile", "shapefile_dir": "/app/effis_layer", "total_features": 8, "saved_geojson": "/tmp/fire_maps/burnt_areas_2021-08_2ba6381e.geojson", "buffer_km": 0.0, "fires": [{"bbox": "23.1766,38.7022,23.4614,3...



model says: The largest fire in Evia in August 2021 was in the north of the island, matching your query. It burned 51,881 ha (EFFIS — Burnt Area Polygons, 2021-08, local shapefile) and started on August 3, 2021. I will use this fire's bbox (23.1766,38.7022,23.4614,39.0379), area and date in the next steps.
MODEL CHOSE:  get_burn_fuel_types   (server: fire-emissions)
  parameters: {"bbox": "23.1766,38.7022,23.4614,39.0379", "fire_date": "2021-08-03"}


  result:     {"bbox": "23.1766,38.7022,23.4614,39.0379", "worldcover_year": 2020, "map_choice": "2020 map: the land cover before a fire in 2021 or earlier.", "source": "ESA WorldCover 2020 v100 (10 m), Microsoft Planetary Computer", "box_hectares_by_class": {"tree_cover": 53205, "water": 27532, "grassland": 5231...



MODEL CHOSE:  estimate_fire_emissions   (server: fire-emissions)
  parameters: {"burned_area_ha": 51881, "bbox": "23.1766,38.7022,23.4614,39.0379", "fuel_shares_pct": {"tree_cover": 85.2, "grassland": 8.4, "shrubland": 3.4, "cropland": 3, "herbaceous_wetland": 0.1}}
  result:     {"burned_area_ha": 51881, "biome": "temperate", "fuel_shares_pct_used": {"tree_cover": 85.1, "grassland": 8.4, "shrubland": 3.4, "cropland": 3.0, "herbaceous_wetland": 0.1}, "emissions_tonnes": {"CO2": {"low": 1090000.0, "central": 2180000.0, "high": 3640000.0}, "CO": {"low": 73000.0, "central": 146...



MODEL CHOSE:  get_smoke_signal   (server: fire-emissions)
  parameters: {"bbox": "23.1766,38.7022,23.4614,39.0379", "fire_date": "2021-08-03", "days_after": 10, "baseline_days": 14}


  result:     {"location": {"lat": 38.87, "lon": 23.319}, "fire_period": "2021-08-03 to 2021-08-13", "baseline_period": "2021-07-20 to 2021-08-02", "pollutants": {"carbon_monoxide": {"unit": "ug/m3", "baseline_mean": 154.8, "fire_period_mean": 323.2, "fire_period_peak": 1909.0, "peak_time_utc": "2021-08-06T21:00"...



MODEL FINAL ANSWER:

### Briefing: Climate Impact of the August 2021 North Evia Wildfire

#### Emissions
The fire burned **51,881 ha** (EFFIS — Burnt Area Polygons, 2021-08, local shapefile) in a temperate biome dominated by tree cover (85.2%), with smaller shares of grassland (8.4%), shrubland (3.4%), and cropland (3.0%) (ESA WorldCover 2020 v100, Microsoft Planetary Computer).

- **CO₂**: **1,090,000–3,640,000 tonnes** (central: **2,180,000 tonnes**).
- **Carbon monoxide (CO)**: **73,000–244,000 tonnes** (central: **146,000 tonnes**).
- **Methane (CH₄)**: **3,190–10,600 tonnes** (central: **6,370 tonnes**).

The CO₂-equivalent (including CO₂, CH₄, and N₂O) is **1,220,000–4,080,000 tonnes** (central: **2,450,000 tonnes**), equivalent to **237,000–790,000 passenger cars** driven for a year (US EPA: 4.6 t CO₂ per typical passenger car per year).

#### Smoke Signal
The fire caused a clear spike in air pollutants near the fire:
- **Carbon monoxide (CO)**: Peak of **1,909 µg/m³** (12.3× ab

## Clean up

Stop our background server.

In [20]:
proc.terminate()
proc.wait(timeout=10)
print("stopped")

stopped


## Summary

| | Already exists | We add |
|---|---|---|
| Server | `effis` (registry, hosted by EVE) | `fire-emissions` (`examples/fire-emissions-server/server.py`) |
| Answers | where, how big, when, burn severity (mapped fires, mostly Europe) | burned area anywhere (MODIS), fuel types, tonnes of CO₂/CO/CH₄/N₂O/PM2.5, smoke vs normal |
| Hand-off | outputs `bbox`, `area_ha`, `firedate` | takes `bbox`, `area_ha`, `firedate` as input |

| Level | Who chooses the tool and parameters | How the call travels |
|---|---|---|
| Python function | you | a normal `await function(...)` |
| MCP server object | you | `mcp.call_tool(name, args)` inside Python |
| MCP over HTTP | you | JSON-RPC `tools/call` POST to `/mcp` (ours locally, EFFIS via EVE's gateway) |
| LLM agent | **the model**, from the docstrings and type hints | the model asks; the client runs it through MCP |

That is why the docstrings and type hints matter so much: at the last level they are the only instructions the model has.